In [1]:
# Orbit Wars Strategy Notebook
"""
This notebook is the single submission artifact.

It contains:
- the Kaggle-compatible `agent(obs)` implementation
- physics-aware target prediction
- fleet sizing and safety-aware launch rules
- an optional local benchmark helper

The workflow is notebook-first so the final Kaggle submission can stay as one `.ipynb`.
"""

'\nThis notebook is the single submission artifact.\n\nIt contains:\n- the Kaggle-compatible `agent(obs)` implementation\n- physics-aware target prediction\n- fleet sizing and safety-aware launch rules\n- an optional local benchmark helper\n\nThe workflow is notebook-first so the final Kaggle submission can stay as one `.ipynb`.\n'

In [11]:
import math
from statistics import mean

try:
    from kaggle_environments import make
except Exception:
    make = None

CENTER_X = 50.0
CENTER_Y = 50.0
SUN_RADIUS = 10.0
MAX_FLEET_SPEED = 6.0
MIN_GARRISON = 10
THREAT_LOOKAHEAD = 10.0
SUN_SAFE_MARGIN = 0.5


def distance(x1, y1, x2, y2):
    return math.hypot(x1 - x2, y1 - y2)


def fleet_speed_for_ships(num_ships, max_speed=MAX_FLEET_SPEED):
    if num_ships <= 1:
        return 1.0
    ships = min(max(num_ships, 1), 1000)
    ratio = math.log(ships) / math.log(1000)
    return 1.0 + (max_speed - 1.0) * (ratio ** 1.5)


def is_orbiting_planet(planet):
    return distance(planet[2], planet[3], CENTER_X, CENTER_Y) + planet[4] < 50.0


def segment_intersects_circle(x1, y1, x2, y2, cx, cy, radius):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return distance(x1, y1, cx, cy) <= radius

    projection = ((cx - x1) * dx + (cy - y1) * dy) / (dx * dx + dy * dy)
    projection = max(0.0, min(1.0, projection))
    closest_x = x1 + projection * dx
    closest_y = y1 + projection * dy
    return distance(closest_x, closest_y, cx, cy) <= radius


def path_crosses_sun(source_x, source_y, target_x, target_y):
    return segment_intersects_circle(
        source_x,
        source_y,
        target_x,
        target_y,
        CENTER_X,
        CENTER_Y,
        SUN_RADIUS + SUN_SAFE_MARGIN,
    )


def get_fleets(obs):
    if isinstance(obs, dict):
        return obs.get('fleets', []), set(obs.get('comet_planet_ids', []))
    return getattr(obs, 'fleets', []), set(getattr(obs, 'comet_planet_ids', []) or [])


def estimate_incoming_threat(planets, fleets, player_id, planet):
    planet_x, planet_y = planet[2], planet[3]
    planet_radius = planet[4]
    threat = 0.0

    for fleet in fleets:
        if fleet[1] == player_id:
            continue
        fleet_x, fleet_y = fleet[2], fleet[3]
        dist = distance(fleet_x, fleet_y, planet_x, planet_y)
        if dist <= planet_radius + THREAT_LOOKAHEAD:
            threat += fleet[6] * 0.75
        elif dist <= planet_radius + 2.0 * THREAT_LOOKAHEAD:
            threat += fleet[6] * 0.35

    return threat


def launch_budget(source_planet, threat=0.0):
    ships = source_planet[5]
    production = source_planet[6]
    reserve = MIN_GARRISON + int(threat) + max(2, production)
    return max(0, ships - reserve)


def predict_planet_position(source_x, source_y, target, angular_velocity, launch_ships, iterations=4):
    target_x, target_y = target[2], target[3]
    if not is_orbiting_planet(target):
        return target_x, target_y

    orbit_dx = target_x - CENTER_X
    orbit_dy = target_y - CENTER_Y
    orbit_radius = math.hypot(orbit_dx, orbit_dy)
    if orbit_radius == 0:
        return target_x, target_y

    current_theta = math.atan2(orbit_dy, orbit_dx)
    travel_time = 0.0
    for _ in range(iterations):
        future_theta = current_theta + angular_velocity * travel_time
        future_x = CENTER_X + orbit_radius * math.cos(future_theta)
        future_y = CENTER_Y + orbit_radius * math.sin(future_theta)
        speed = fleet_speed_for_ships(launch_ships)
        travel_time = distance(source_x, source_y, future_x, future_y) / max(speed, 1e-6)

    final_theta = current_theta + angular_velocity * travel_time
    return CENTER_X + orbit_radius * math.cos(final_theta), CENTER_Y + orbit_radius * math.sin(final_theta)


def target_priority(source_x, source_y, target, comet_ids=None):
    comet_ids = comet_ids or set()
    if target[0] in comet_ids:
        return -10.0

    dist = distance(source_x, source_y, target[2], target[3])
    production = target[6]
    ships = target[5]
    owner_bonus = 0.7 if target[1] == -1 else 0.45
    orbit_bonus = 1.15 if is_orbiting_planet(target) else 1.0
    return orbit_bonus * (production + owner_bonus) / (dist + 1.0) - 0.015 * ships


In [12]:
def simple_attacker(obs, conf=None):
    if isinstance(obs, dict):
        player_id = obs.get('player', 0)
        planets = obs.get('planets', [])
        angular_velocity = obs.get('angular_velocity', 0.05)
    else:
        player_id = obs.player
        planets = obs.planets
        angular_velocity = getattr(obs, 'angular_velocity', 0.05)

    moves = []
    owned_planets = [planet for planet in planets if planet[1] == player_id]
    if not owned_planets:
        return moves

    enemy_targets = [planet for planet in planets if planet[1] != player_id]
    if not enemy_targets:
        return moves

    for source in owned_planets:
        source_id, _, source_x, source_y, _, source_ships, _ = source
        max_launch = max(0, source_ships - MIN_GARRISON)
        if max_launch <= 0:
            continue

        ranked_targets = sorted(
            enemy_targets,
            key=lambda planet: target_priority(source_x, source_y, planet),
            reverse=True,
        )

        launch_ships = None
        launch_angle = None

        for target in ranked_targets:
            tentative_launch = max(1, max_launch // 2)
            predicted_x, predicted_y = predict_planet_position(
                source_x,
                source_y,
                target,
                angular_velocity,
                tentative_launch,
            )
            travel_speed = fleet_speed_for_ships(tentative_launch)
            travel_time = distance(source_x, source_y, predicted_x, predicted_y) / max(travel_speed, 1e-6)
            ships_needed = int(target[5] + target[6] * travel_time + 1)

            if ships_needed <= max_launch:
                launch_ships = ships_needed
                predicted_x, predicted_y = predict_planet_position(
                    source_x,
                    source_y,
                    target,
                    angular_velocity,
                    launch_ships,
                )
                launch_angle = math.atan2(predicted_y - source_y, predicted_x - source_x)
                break

        if launch_ships is not None and launch_angle is not None:
            moves.append([source_id, launch_angle, launch_ships])

    return moves


def agent(obs, conf=None):
    return simple_attacker(obs, conf)


In [9]:
def baseline_agent(obs, conf=None):
    if isinstance(obs, dict):
        player_id = obs.get('player', 0)
        planets = obs.get('planets', [])
    else:
        player_id = obs.player
        planets = obs.planets

    moves = []
    owned_planets = [planet for planet in planets if planet[1] == player_id]
    if not owned_planets:
        return moves

    targets = [planet for planet in planets if planet[1] != player_id]
    if not targets:
        return moves

    for source in owned_planets:
        source_id, _, source_x, source_y, _, source_ships, _ = source
        if source_ships <= MIN_GARRISON + 5:
            continue
        nearest = min(targets, key=lambda target: distance(source_x, source_y, target[2], target[3]))
        ships_needed = nearest[5] + 1
        if source_ships >= ships_needed:
            launch_angle = math.atan2(nearest[3] - source_y, nearest[2] - source_x)
            moves.append([source_id, launch_angle, ships_needed])

    return moves


def run_local_benchmark(num_games=20, seed=42):
    if make is None:
        print('kaggle_environments is not available in this kernel.')
        return None

    env = make('orbit_wars', configuration={'seed': seed}, debug=False)
    agent_wins = 0
    baseline_wins = 0
    draws = 0

    agent_scores = []
    baseline_scores = []

    for game_idx in range(num_games):
        if game_idx % 2 == 0:
            agents = [agent, baseline_agent]
            agent_index = 0
            baseline_index = 1
        else:
            agents = [baseline_agent, agent]
            agent_index = 1
            baseline_index = 0

        env.run(agents)
        final = env.steps[-1]
        agent_score = getattr(final[agent_index], 'reward', 0) or 0
        baseline_score = getattr(final[baseline_index], 'reward', 0) or 0

        agent_scores.append(agent_score)
        baseline_scores.append(baseline_score)

        if agent_score > baseline_score:
            agent_wins += 1
            result = 'AGENT WIN'
        elif baseline_score > agent_score:
            baseline_wins += 1
            result = 'BASELINE WIN'
        else:
            draws += 1
            result = 'DRAW'

        print(f'Game {game_idx + 1:02d}: {result} | Agent: {agent_score} vs Baseline: {baseline_score}')

    win_rate = (agent_wins / num_games) * 100 if num_games else 0.0
    print('\nBenchmark summary')
    print('-' * 40)
    print(f'Agent win rate: {win_rate:.1f}%')
    print(f'Record: {agent_wins} wins | {baseline_wins} losses | {draws} draws')
    print(f'Average score: {mean(agent_scores) if agent_scores else 0.0:.1f} vs {mean(baseline_scores) if baseline_scores else 0.0:.1f}')

    return {
        'agent_wins': agent_wins,
        'baseline_wins': baseline_wins,
        'draws': draws,
        'agent_scores': agent_scores,
        'baseline_scores': baseline_scores,
        'win_rate': win_rate,
    }


# Uncomment to benchmark locally after opening the notebook kernel.
# run_local_benchmark(num_games=20)
